# Notebook 01 — Model Training: YOLO + EfficientNet

This notebook walks through:
1. Dataset preparation and augmentation
2. YOLOv9-C fine-tuning for 16-class damage detection
3. EfficientNet-B4 fine-tuning for damage severity classification
4. Model evaluation and threshold selection
5. ONNX export for production deployment

**Prerequisites:**
```bash
pip install ultralytics timm albumentations torchvision onnx onnxsim
```

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Dataset Preparation

In [ ]:
DATASET_ROOT = PROJECT_ROOT / 'datasets' / 'damage_detection'
print(f'Dataset path: {DATASET_ROOT}')

# Count images in each split
for split in ['train', 'val', 'test']:
    img_dir = DATASET_ROOT / 'images' / split
    if img_dir.exists():
        n = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
        print(f'  {split}: {n} images')
    else:
        print(f'  {split}: directory not found')

In [ ]:
from src.vision.detector import DAMAGE_CLASSES

# Class distribution analysis
label_counts = {cls: 0 for cls in DAMAGE_CLASSES}

label_dir = DATASET_ROOT / 'labels' / 'train'
if label_dir.exists():
    for label_file in label_dir.glob('*.txt'):
        with open(label_file) as f:
            for line in f:
                class_idx = int(line.split()[0])
                if class_idx < len(DAMAGE_CLASSES):
                    label_counts[DAMAGE_CLASSES[class_idx]] += 1

    df_counts = pd.Series(label_counts).sort_values(ascending=False)
    df_counts.plot(kind='bar', figsize=(14, 5), color='steelblue')
    plt.title('Damage Class Distribution (Training Set)')
    plt.xlabel('Damage Class')
    plt.ylabel('Annotation Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Label directory not found — skipping distribution plot')

## 2. YOLOv9-C Fine-Tuning

In [ ]:
from ultralytics import YOLO

DATA_YAML = str(DATASET_ROOT / 'data.yaml')
EPOCHS = 100
BATCH = 16
IMG_SIZE = 640

model = YOLO('yolov9c.pt')  # pre-trained on COCO

if Path(DATA_YAML).exists():
    results = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        batch=BATCH,
        imgsz=IMG_SIZE,
        device=0 if torch.cuda.is_available() else 'cpu',
        project=str(PROJECT_ROOT / 'runs' / 'train'),
        name='damage_yolov9',
        patience=50,
        mosaic=1.0,
        mixup=0.1,
        lr0=0.01,
    )
else:
    print(f'data.yaml not found at {DATA_YAML}')
    print('Create the dataset structure first (see docs/camera_setup.md)')

In [ ]:
# Load training curves from results CSV
results_csv = PROJECT_ROOT / 'runs' / 'train' / 'damage_yolov9' / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    for ax, col in zip(axes.flatten(), 
                       ['train/box_loss', 'train/cls_loss', 'train/dfl_loss',
                        'metrics/mAP50', 'metrics/mAP50-95', 'val/box_loss']):
        if col in df.columns:
            df[col].plot(ax=ax, title=col)
    plt.tight_layout()
    plt.show()

## 3. EfficientNet-B4 Damage Severity Classifier

In [ ]:
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, ImageFolder

# EfficientNet-B4 for damage severity classification
# Expected dataset structure:
#   datasets/damage_severity/
#     train/none/   val/none/
#     train/minor/  val/minor/
#     train/moderate/ val/moderate/
#     train/severe/   val/severe/

SEVERITY_DATASET = PROJECT_ROOT / 'datasets' / 'damage_severity'
NUM_CLASSES = 4  # none, minor, moderate, severe
EPOCHS_CLS = 50
LR_CLS = 1e-4

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('Dataset layout prepared')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Build EfficientNet-B4 with custom head
classifier = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
in_features = classifier.classifier[1].in_features
classifier.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, NUM_CLASSES)
)
classifier = classifier.to(device)

# Freeze backbone initially, train only head
for param in classifier.features.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in classifier.parameters())
trainable_params = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable params (head only): {trainable_params:,}')

## 4. Model Evaluation

In [ ]:
# Evaluate trained YOLO on validation set
best_weights = PROJECT_ROOT / 'runs' / 'train' / 'damage_yolov9' / 'weights' / 'best.pt'
if best_weights.exists():
    trained_model = YOLO(str(best_weights))
    if Path(DATA_YAML).exists():
        metrics = trained_model.val(data=DATA_YAML)
        print(f'mAP50: {metrics.box.map50:.3f}')
        print(f'mAP50-95: {metrics.box.map:.3f}')
        print(f'Precision: {metrics.box.mp:.3f}')
        print(f'Recall: {metrics.box.mr:.3f}')
else:
    print('No trained weights found — run training first')

## 5. ONNX Export

In [ ]:
# Export best YOLO to ONNX for production deployment
if best_weights.exists():
    output_onnx = str(PROJECT_ROOT / 'models' / 'weights' / 'yolo_damage.onnx')
    trained_model.export(format='onnx', imgsz=640, opset=17, simplify=True)
    print(f'ONNX model exported')

    # Copy best.pt to models/weights/
    import shutil
    dest = PROJECT_ROOT / 'models' / 'weights' / 'yolo_damage.pt'
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(best_weights), str(dest))
    print(f'Best weights copied → {dest}')